In [ ]:
# SETUP - safe to re-run. Re-clones the CODE only; never touches result files.
%cd /kaggle/working
!rm -rf StatTrap_SLM
!git clone https://github.com/mirzayasirabdullahbaig07/StatTrap_SLM.git
%cd /kaggle/working/StatTrap_SLM
!git log -1 --oneline
!grep -q "direct_mode" src/stattrap/cli.py && echo "CODE OK: latest version" || echo "!!! OLD CODE ON GITHUB - push your latest commit from your PC, then re-run this cell"
!echo "commit above must be the one you just pushed" 
!pip install -q -e . accelerate bitsandbytes

In [ ]:
# RUN - resumable: if the session stops or errors, run this SAME cell again (same --out file)
%cd /kaggle/working/StatTrap_SLM
!stattrap run --backend hf --load-in-4bit --per-family 30 --perms 2 --max-new-tokens 1024 --direct-mode logit --batch-size 8 --out /kaggle/working/nb2_qwen7b.csv --models Qwen/Qwen2.5-7B-Instruct

In [ ]:
# DONE? Zip result + manifest, then download nb2_qwen7b.zip from the Output panel (right side)
!cd /kaggle/working && zip -q nb2_qwen7b.zip nb2_qwen7b.csv nb2_qwen7b.manifest.json && ls -la nb2_qwen7b.zip
import pandas as pd
d = pd.read_csv('/kaggle/working/nb2_qwen7b.csv', keep_default_na=False)
print(d.groupby(['model','style']).agg(rows=('correct','size'), acc=('correct','mean'), unparsed=('chosen', lambda s: (s=='').mean()), trunc=('truncated','mean')).round(3))